In [14]:
import math
import numpy as np
import pandas as pd

In [15]:
alpha = 0
y0, z0 = 1, 4
x_start = np.array([1.0, 2.0, 5.0])

In [16]:
def rotated_coords(y, z):
    """Преобразование координат с поворотом и смещением"""
    y_p = (y - y0) * math.cos(alpha) + (z - z0) * math.sin(alpha)
    z_p = -(y - y0) * math.sin(alpha) + (z - z0) * math.cos(alpha)
    return y_p, z_p

In [17]:
def f3(x, y, z):
    """Целевая функция для минимизации"""
    y_p, z_p = rotated_coords(y, z)
    return 2 * x**4 + 2 * y_p**4 + z_p**2

In [18]:
def numerical_gradient(f, x, eps=1e-8):
    """Численное вычисление градиента для любой функции"""
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[i] += eps
        x_minus[i] -= eps
        grad[i] = (f(*x_plus) - f(*x_minus)) / (2 * eps)
    return grad

In [19]:
def grad_f3(x, y, z):
    """Вычисление градиента целевой функции (универсальная версия)"""
    point = np.array([x, y, z])
    return numerical_gradient(f3, point)

In [20]:
def numerical_hessian(f, x, eps=1e-6):
    """Численное вычисление матрицы Гессе для любой функции в точке x"""
    n = len(x)
    H = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            # Вторая производная по i и j переменным
            x_pp = x.copy(); x_pp[i] += eps; x_pp[j] += eps
            x_pm = x.copy(); x_pm[i] += eps; x_pm[j] -= eps
            x_mp = x.copy(); x_mp[i] -= eps; x_mp[j] += eps
            x_mm = x.copy(); x_mm[i] -= eps; x_mm[j] -= eps
            
            H[i, j] = (f(*x_pp) - f(*x_pm) - f(*x_mp) + f(*x_mm)) / (4 * eps**2)
    
    return H

In [21]:
def hessian_f3(x, y, z):
    """Вычисление матрицы Гессе (вторых производных) численным методом"""
    point = np.array([x, y, z])
    return numerical_hessian(f3, point)

In [22]:
def golden_section(phi, a=-1, b=1, tol=1e-10):
    """Одномерная минимизация методом золотого сечения"""
    multiplier = (math.sqrt(5) - 1) / 2
    x1 = b - (b - a) * multiplier
    x2 = a + (b - a) * multiplier
    f1 = phi(x1)
    f2 = phi(x2)
    iterations = 0
    max_iter = 100
    while abs(b - a) > tol and iterations < max_iter:
        if f1 < f2:
            b = x2
            x2 = x1
            f2 = f1
            x1 = b - multiplier * (b - a)
            f1 = phi(x1)
        else:
            a = x1
            x1 = x2
            f1 = f2
            x2 = a + multiplier * (b - a)
            f2 = phi(x2)
        iterations += 1
    return (a + b) / 2

## Метод координатного спуска

**Алгоритм:**

1. **Инициализация:** Начальная точка $x^{(0)} = x_0$
2. **Итерационный процесс:** Для $k = 0, 1, 2, \ldots$
   
   Для каждой координаты $i = 1, 2, 3$:
   - Решаем одномерную задачу оптимизации:
     $$
     \alpha^* = \arg\min_{\alpha} f(x_1^{(k)}, \ldots, x_i^{(k)} + \alpha, \ldots, x_3^{(k)})
     $$
   - Обновляем координату:
     $$
     x_i^{(k+1)} = x_i^{(k)} + \alpha^*
     $$

3. **Критерий остановки:**
   $$
   \|x^{(k+1)} - x^{(k)}\| < \varepsilon
   $$

**Математическая запись:**
$$
\begin{align*}
x_i^{(k+1)} &= \arg\min_{t} f(x_1^{(k+1)}, \ldots, x_{i-1}^{(k+1)}, t, x_{i+1}^{(k)}, \ldots, x_n^{(k)}) \\
&= x_i^{(k)} + \alpha^*
\end{align*}
$$

**Особенности:**
- Одномерная минимизация выполняется методом золотого сечения
- На каждой итерации циклически обновляются все координаты
- Простая реализация, но может медленно сходиться для "овражных" функций

In [23]:
def coordinate_descent(x0, eps=1e-8, max_iter=1000):
    """Минимизация методом координатного спуска"""
    x = np.array(x0, dtype=float)
    for _ in range(max_iter):
        x_old = x.copy()
        # Поочередная оптимизация по каждой координате
        for i in range(3):
            def phi(alpha_val):
                x_temp = x.copy()
                x_temp[i] += alpha_val
                return f3(*x_temp)
            alpha_opt = golden_section(phi, -0.5, 0.5, tol=eps/10)
            x[i] += alpha_opt
        # Проверка условия сходимости
        if np.linalg.norm(x - x_old) < eps:
            break
    return x

## Метод наискорейшего спуска

**Алгоритм:**

1. **Инициализация:** Начальная точка $x^{(0)} = x_0$
2. **Итерационный процесс:** Для $k = 0, 1, 2, \ldots$
   - Вычисляем градиент: $\nabla f(x^{(k)})$
   - Проверяем условие остановки: $\|\nabla f(x^{(k)})\| < \varepsilon$
   - Определяем направление спуска (антиградиент):
     $$
     p^{(k)} = -\frac{\nabla f(x^{(k)})}{\|\nabla f(x^{(k)})\|}
     $$
   - Решаем задачу одномерной минимизации:
     $$
     \alpha^* = \arg\min_{\alpha > 0} f(x^{(k)} + \alpha p^{(k)})
     $$
   - Обновляем точку:
     $$
     x^{(k+1)} = x^{(k)} + \alpha^* p^{(k)}
     $$

**Математическая формулировка:**
$$
x^{(k+1)} = x^{(k)} - \alpha_k \frac{\nabla f(x^{(k)})}{\|\nabla f(x^{(k)})\|}
$$

где $\alpha_k$ находится из условия:
$$
\alpha_k = \arg\min_{\alpha > 0} f\left(x^{(k)} - \alpha \frac{\nabla f(x^{(k)})}{\|\nabla f(x^{(k)})\|}\right)
$$

**Особенности:**
- Направление спуска - нормализованный антиградиент
- Длина шага определяется точным поиском (золотое сечение)
- Ограничение максимального шага: $\alpha_{\text{max}} = \min\left(1, \frac{0.1}{\|\nabla f\|}\right)$
- Гарантированная сходимость для выпуклых функций

In [24]:
def steepest_descent(x0, eps=1e-8, max_iter=1000):
    """Минимизация методом наискорейшего спуска"""
    x = np.array(x0, dtype=float)
    for _ in range(max_iter):
        grad = grad_f3(*x)
        grad_norm = np.linalg.norm(grad)
        if grad_norm < eps:
            break
        # Направление антиградиента
        p = -grad / grad_norm
        def phi(alpha_val):
            return f3(*(x + alpha_val * p))
        max_step = min(1.0, 0.1 / (grad_norm + 1e-10))
        alpha_opt = golden_section(phi, 0, max_step, tol=eps/10)
        x = x + alpha_opt * p
    return x

## Метод Ньютона

**Алгоритм:**

1. **Инициализация:** Начальная точка $x^{(0)} = x_0$
2. **Итерационный процесс:** Для $k = 0, 1, 2, \ldots$
   - Вычисляем градиент: $g^{(k)} = \nabla f(x^{(k)})$
   - Проверяем условие остановки: $\|g^{(k)}\| < \varepsilon$
   - Вычисляем матрицу Гессе: $H^{(k)} = \nabla^2 f(x^{(k)})$
   - Решаем систему Ньютона:
     $$
     H^{(k)} \Delta x^{(k)} = -g^{(k)}
     $$
   - Если $H^{(k)}$ вырождена ($|\det(H^{(k)})| < 10^{-12}$), используем псевдообратную матрицу
   - Ограничиваем шаг: если $\|\Delta x^{(k)}\| > 10$, то $\Delta x^{(k)} = 10 \cdot \frac{\Delta x^{(k)}}{\|\Delta x^{(k)}\|}$
   - Обновляем точку:
     $$
     x^{(k+1)} = x^{(k)} + \Delta x^{(k)}
     $$

**Математическая формулировка:**
$$
x^{(k+1)} = x^{(k)} - [H^{(k)}]^{-1} \nabla f(x^{(k)})
$$

где:
- $H^{(k)} = \nabla^2 f(x^{(k)})$ - матрица Гессе
- $\nabla f(x^{(k)})$ - градиент функции

**Особенности:**
- Квадратичная скорость сходимости вблизи решения
- Требует вычисления вторых производных (матрицы Гессе)
- Защита от вырожденной матрицы Гессе через псевдообратную матрицу
- Ограничение шага для устойчивости при удаленных начальных приближениях
- Может сходиться к седловым точкам

In [25]:
def newton_method(x0, eps=1e-10, max_iter=1000):
    """Минимизация методом Ньютона"""
    x = np.array(x0, dtype=float)
    for _ in range(max_iter):
        grad = grad_f3(*x)
        grad_norm = np.linalg.norm(grad)
        if grad_norm < eps:
            break
        H = hessian_f3(*x)
        # Проверка вырожденности матрицы Гессе
        det_H = np.linalg.det(H)
        if abs(det_H) < 1e-12:
            dx = -np.linalg.pinv(H) @ grad
        else:
            dx = -np.linalg.solve(H, grad)
        # Ограничение максимального шага
        step_norm = np.linalg.norm(dx)
        if step_norm > 10:
            dx = dx / step_norm * 10
        x = x + dx
    return x

In [26]:
coord_res = coordinate_descent(x_start)
steep_res = steepest_descent(x_start)
newton_res = newton_method(x_start)

data = []
methods = [
    ("Координатный спуск", coord_res),
    ("Наискорейший спуск", steep_res),
    ("Метод Ньютона", newton_res),
    ("Аналитический минимум", np.array([0.0, 1.0, 4.0]))
]

for name, res in methods:
    data.append({
        "Метод": name,
        "x": res[0],
        "y": res[1], 
        "z": res[2],
        "f(x,y,z)": f3(*res)
    })

df = pd.DataFrame(data)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

df

,Метод,x,y,z,"f(x,y,z)"
0,Координатный спуск,8.811326e-10,1.000000,4.000000,1.536104e-36
1,Наискорейший спуск,6.207017e-03,1.006207,3.999999,5.937731e-09
2,Метод Ньютона,2.004854e-04,1.000200,4.000000,6.462618e-15
3,Аналитический минимум,0.000000e+00,1.000000,4.000000,0.000000e+00
